# COSC2670/2738 Assignment 3 — Framework

**Important:** Please read the comments carefully. Insert your code only in the designated cells.
Do **not** modify cells marked with `# Please don't change this cell`.

Changing locked cells may cause errors during automatic marking and invalidate your submission.

## Install and load necessary packages

In [1]:
# Please don't change this cell

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

## Load the MovieLens 100K dataset

You must use `u1.base` / `u1.test` provided in the dataset and use it consistently across all tasks. Do not create your own random split.


In [2]:
# You may change the file paths to use a different split pair
TRAIN_PATH = 'ml-100k/u1.base'
TEST_PATH = 'ml-100k/u1.test'

names = ['user_id', 'item_id', 'rating', 'timestamp']
train_df = pd.read_csv(TRAIN_PATH, sep='\t', names=names)
test_df = pd.read_csv(TEST_PATH, sep='\t', names=names)

n_users = 943
n_items = 1682

print(f'{n_users} users')
print(f'{n_items} items')
print(f'Training ratings: {len(train_df)}')
print(f'Test ratings: {len(test_df)}')

FileNotFoundError: [Errno 2] No such file or directory: 'ml-100k/u1.base'

## Construct the user-item rating matrices

In [ ]:
# Please don't change this cell

# Build training rating matrix (n_users x n_items), 0 means unrated
train_matrix = np.zeros((n_users, n_items))
for row in train_df.itertuples():
    train_matrix[row.user_id - 1, row.item_id - 1] = row.rating

# Build test rating matrix
test_matrix = np.zeros((n_users, n_items))
for row in test_df.itertuples():
    test_matrix[row.user_id - 1, row.item_id - 1] = row.rating

print("Training rating matrix shape:", train_matrix.shape)
print("Test rating matrix shape:", test_matrix.shape)

## MAE and RMSE evaluation utilities

In [ ]:
# Please don't change this cell

EPSILON = 1e-9

def evaluate(test_matrix, predicted_matrix):
    '''
    Evaluate rating prediction using MAE and RMSE.
    Only considers entries where test_matrix > 0 (i.e. actual test ratings).
    '''
    mask = test_matrix > 0
    n_test = np.sum(mask)
    if n_test == 0:
        return 0.0, 0.0
    errors = test_matrix[mask] - predicted_matrix[mask]
    MAE = np.mean(np.abs(errors))
    RMSE = np.sqrt(np.mean(errors ** 2))
    return MAE, RMSE


---
# Task 1: Reproduce the Sarwar Item-Based CF Baseline Model (10 marks)

Implement the item-based collaborative filtering method from Sarwar et al. (2001) with:
- **Similarity:** Adjusted cosine similarity (Section 3.1.3)
- **Prediction:** KNN-regression as taught in lectures (k = 20)

Your code must:
1. Compute the adjusted cosine item-item similarity matrix using the training data
2. For each test rating, predict the rating using KNN-regression with k=20 neighbours
3. Store all predictions in a `predicted_matrix` (same shape as `test_matrix`)
4. Save the MAE and RMSE into the variables below

In [ ]:
# Write your code here for Task 1
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np


#user means
user_means = np.zeros(n_users)

for u in range (n_users):
    ratings = train_matrix[u]
    rated = ratings[ratings > 0]

    if len(rated) > 0:
        user_means[u] = np.mean(rated)
    else:
        user_means[u] = 3.0
        
#adjusted ratings
adjusted_matrix = np.zeros((n_users, n_items))

for u in range(n_users):
    for i in range(n_items):
        if train_matrix[u,i] > 0:
            adjusted_matrix[u, i] = train_matrix[u, i] - user_means[u]

#adjusted cosine similarity
item_similarity = np.zeros((n_items, n_items))
                           
for i in range(n_items):
    for j in range(i, n_items):


        vec_i = adjusted_matrix[:,i]
        vec_j = adjusted_matrix[:,j]

        numerator = np.sum(vec_i * vec_j)

        denominator = (
            np.sqrt(np.sum(vec_i ** 2))
            * np.sqrt(np.sum(vec_j**2))
        )

        if denominator > 0:
            sim = numerator/denominator
        else:
            sim = 0

        item_similarity[i,j] = sim
        item_similarity[j,i] = sim

# KNN regression prediction (k=20)
k = 20

YOUR_PREDICTED_MATRIX = np.zeros((n_users, n_items))

for u in range(n_users):
    rated_items = np.where(train_matrix[u] > 0)[0]

    for i in range(n_items):

        if len(rated_items)==0:
            YOUR_PREDICTED_MATRIX[u, i] = user_means[u]
            continue

        sims = item_similarity[i, rated_items]

        top_idx = np.argsort(np.abs(sims))[-20:]
        neighbours = rated_items[top_idx]
        neighbour_sims = sims[top_idx]
        neighbour_ratings = train_matrix[u, neighbours]

        denominator = np.sum(np.abs(neighbour_sims))
                                            

        if denominator > 0:
            neighbour_deviations = neighbour_ratings - user_means[u]
            prediction = user_means[u] + np.sum(neighbour_sims * neighbour_deviations) / denominator
        else:
            prediction = user_means[u]

        YOUR_PREDICTED_MATRIX[u, i] = min(5, max(1, prediction))


In [ ]:
MAE_task1,RMSE_task1 = evaluate(test_matrix, YOUR_PREDICTED_MATRIX)

In [ ]:
# Please don't change this cell
print("===================== The MAE and RMSE of Your Implementation =====================")
print("Task 1: Baseline Item-Based CF (Adjusted Cosine + KNN-Regression, k=20)")
print("MAE: {}, RMSE: {}" .format(MAE_task1, RMSE_task1))
print("=" * 70)

The Baseline recommender system was implemented using adjusted cosine similarity and KNN regression with k=20. The model achieved an MAE of 0.7501 and an RMSE of 0.9611 on the test dataset. These results provide the reference  point for the controlled improvement experiments conducted in Task2.

---
# Task 2: Controlled Improvement Experiments (10 marks)

Choose **One** options from A and B, and **One** from C and D below. Each experiment is independent.
Except for the specific parameter being tested, all other settings must remain the same as the baseline (adjusted cosine, KNN-regression, k=20).

**You must indicate which two options you chose.**

In [ ]:
# Please specify which two options you selected (e.g. 'A', 'B', 'C', or 'D')
OPTION_1 = 'A'  # e.g. 'A'
OPTION_2 = 'C'  # e.g. 'C'

## Task 2 — Option A: Similarity Measure Comparison

Replace the baseline adjusted cosine similarity with **cosine similarity** and **Pearson correlation**.
Keep k=20 and KNN-regression. Report MAE and RMSE for each.

**Skip this section if you did not select Option A.**

In [ ]:
# Write your code here for Option A (if selected)
# 1. Compute cosine similarity -> predict -> evaluate
# 2. Compute Pearson correlation similarity -> predict -> evaluate
def compute_cosine_similarity(matrix):
    sim_matrix = np.zeros((n_items, n_items))

    for i in range(n_items):
        for j in range(i, n_items):
            vec_i = matrix[:, i]
            vec_j = matrix[:, j]

            numerator = np.sum(vec_i * vec_j)
            denominator = np.sqrt(np.sum(vec_i ** 2)) * np.sqrt(np.sum(vec_j ** 2))

            if denominator > 0:
                sim = numerator / denominator
            else:
                sim = 0

            sim_matrix[i, j] = sim 
            sim_matrix[j, i] = sim
            
    return sim_matrix

def compute_pearson_similarity(matrix):
    sim_matrix = np.zeros((n_items, n_items))

    for i in range(n_items):
        for j in range(i, n_items):

            users = np.where((matrix[:, i] > 0) & (matrix[:, j] > 0))[0]

            if len(users) > 1:
                ratings_i = matrix[users, i]
                ratings_j = matrix[users, j]

                ratings_i  = ratings_i - np.mean(ratings_i)
                ratings_j  = ratings_j - np.mean(ratings_j)

                numerator = np.sum(ratings_i * ratings_j)
                denominator = np.sqrt(np.sum(ratings_i ** 2)) * np.sqrt(np.sum(ratings_j ** 2))

                if denominator > 0:
                    sim = numerator / denominator
                else:
                    sim = 0

                sim_matrix[i ,j] = sim
                sim_matrix[j, i] = sim
                
    return sim_matrix
            

def predict_with_similarity(similarity_matrix, k=20):
    predicted_matrix = np.zeros((n_users, n_items))

    for u in range(n_users):
        rated_items = np.where(train_matrix[u] > 0)[0]

        for i in range(n_items):

            if len(rated_items) == 0:
                predicted_matrix[u, i] =user_means[u]
                continue

            sims = similarity_matrix[i, rated_items]

            top_idx = np.argsort(np.abs(sims))[-k:]
            neighbours = rated_items[top_idx]

            neighbour_sims = sims[top_idx]
            neighbour_ratings = train_matrix[u, neighbours]

            denominator = np.sum(np.abs(neighbour_sims))

            if denominator > 0:
                prediction = np.sum(neighbour_sims * neighbour_ratings) / denominator
            else:
                prediction = user_means[u]

            predicted_matrix[u, i] = min(5, max(1, prediction))

    return predicted_matrix



In [ ]:
# --- Save your results ---
# cosine similarity
cosine_similarity = compute_cosine_similarity(train_matrix)
cosine_pred_matrix = predict_with_similarity(cosine_similarity, k=20)
MAE_optionA_cosine, RMSE_optionA_cosine = evaluate(test_matrix, cosine_pred_matrix)

#pearson correlation similarity
pearson_similarity = compute_pearson_similarity(train_matrix)
pearson_pred_matrix = predict_with_similarity(pearson_similarity, k=20)
MAE_optionA_pearson, RMSE_optionA_pearson = evaluate(test_matrix, pearson_pred_matrix)
                                                                


In [ ]:
# Please don't change this cell

print("=" * 70)
print("Option A: Similarity Measure Comparison")
print("=" * 70)
print(f"Cosine similarity:      MAE = {MAE_optionA_cosine},  RMSE = {RMSE_optionA_cosine}")
print(f"Pearson correlation:    MAE = {MAE_optionA_pearson},  RMSE = {RMSE_optionA_pearson}")
print(f"Baseline (adj cosine):  MAE = {MAE_task1},  RMSE = {RMSE_task1}")

This experiment compared adjusted cosine similarity, cosine similarity, and pearson correlation while keeping the prediction method and neighbourhood size  unchanged.

The baseline adjusted cosine similarity achieved the MAE (0.7501) and RMSE (0.9611), indicating the best overall recommendation accuracy. 

Cosine similarity produced slightlynhigher error values , suggesting thst it was less effective at handling differences in user rating behaviour. 

Pearson correlation resulted in the highest error values , showing significantly poorer performance. Therefore, adjusted cosine similarity is the most suitable similarity.

## Task 2 — Option B: Prediction Method Comparison

### Not selected (because OPTION_1='A', OPTION_2='C')
Replace KNN-regression with the **weighted sum** method (Section 3.2.1 of Sarwar et al.).
Keep adjusted cosine similarity and k=20.
Report MAE and RMSE.

**Skip this section if you did not select Option B.**



In [ ]:
# Write your code here for Option B (if selected)
# Implement weighted sum prediction and evaluate


In [ ]:
# --- Save your results ---
MAE_optionB, RMSE_optionB = evaluate(test_matrix, YOUR_PREDICTED_MATRIX)

In [ ]:
# Please don't change this cell

print("=" * 70)
print("Option B: Prediction Method Comparison")
print("=" * 70)
print(f"Weighted sum:           MAE = {MAE_optionB},  RMSE = {RMSE_optionB}")
print(f"Baseline (KNN-regr):    MAE = {MAE_task1},  RMSE = {RMSE_task1}")

## Task 2 — Option C: Neighbourhood Size Tuning

Test **k = 5, 10, 20, 30, 50** using the baseline similarity (adjusted cosine) and KNN-regression.
Report MAE and RMSE for each k value.

**Skip this section if you did not select Option C.**

In [ ]:
# Write your code here for Option C (if selected)
# Test k = 5, 10, 20, 30, 50 and record MAE/RMSE for each
def predict_with_k(k):
    print("Running k =",k)
    predicted_matrix = np.zeros((n_users, n_items))

    for u in range(n_users):
        rated_items = np.where(train_matrix[u] >0)[0]

        for i in range(n_items):
            
            if  len(rated_items) == 0:
                predicted_matrix[u, i] = user_means[u]
                continue

            sims = item_similarity[i, rated_items]

            top_idx = np.argsort(np.abs(sims))[-k:]
            neighbours = rated_items[top_idx]
            neighbour_sims = sims[top_idx]
            neighbour_ratings = train_matrix[u, neighbours]

            denominator = np.sum(np.abs(neighbour_sims))
                                            

            if denominator > 0:
                neighbour_deviations = neighbour_ratings - user_means[u]
                prediction = user_means[u] + np.sum(neighbour_sims * neighbour_deviations) / denominator
            else:
                prediction = user_means[u]           
            predicted_matrix[u, i] = min(5, max(1, prediction))

    return predicted_matrix


In [ ]:
# --- Save your results ---
#k = 5 
pred_k5 = predict_with_k(5)
MAE_optionC_k5, RMSE_optionC_k5 = evaluate(test_matrix, pred_k5)

#k = 10
pred_k10 = predict_with_k(10)
MAE_optionC_k10, RMSE_optionC_k10 = evaluate(test_matrix, pred_k10)

#k = 20 
pred_k20 = predict_with_k(20)
MAE_optionC_k20, RMSE_optionC_k20 = evaluate(test_matrix, pred_k20)

#k = 30 
pred_k30 = predict_with_k(30)
MAE_optionC_k30, RMSE_optionC_k30 = evaluate(test_matrix, pred_k30)

#k = 50 
pred_k50 = predict_with_k(50)
MAE_optionC_k50, RMSE_optionC_k50 = evaluate(test_matrix, pred_k50)

In [ ]:
# Please don't change this cell

print("=" * 70)
print("Option C: Neighbourhood Size Tuning")
print("=" * 70)
print(f"k =  5:  MAE = {MAE_optionC_k5},  RMSE = {RMSE_optionC_k5}")
print(f"k = 10:  MAE = {MAE_optionC_k10},  RMSE = {RMSE_optionC_k10}")
print(f"k = 20:  MAE = {MAE_optionC_k20},  RMSE = {RMSE_optionC_k20}")
print(f"k = 30:  MAE = {MAE_optionC_k30},  RMSE = {RMSE_optionC_k30}")
print(f"k = 50:  MAE = {MAE_optionC_k50},  RMSE = {RMSE_optionC_k50}")



The experiment investigated the effect of neighbourhood size on recommendation accuracy using adjusted cosine similarity and KNN regression. The results show the prediction accuracy varied slightly across different values of k. 

The lowest MAE was achieved at k=20, while the lowest RMSE was achieved at k=30. Performance was relatively stable for k values between 20 and 50, indicating that the model is not highly sensitive to neighbourhood size within this range. 

Smaller neighbourhood sizes such as k=5 produced slightly higher prediction errors because fewer neighbouring items were available to contribute to the recommendation. 

Overall, k=20 and k=30 provided the best performance  and are suitable choices for this recommender system.

## Task 2 — Option D: Significance Weighting

### Not selected (because OPTION_1='A', OPTION_2='C')

Apply significance weighting: `weighted_sim(i,j) = sim(i,j) × min(c(i,j), T) / T`

Test **T = 10, 20, 30, 50**. Keep adjusted cosine, KNN-regression, k=20.
Report MAE and RMSE for each T value.

**Skip this section if you did not select Option D.**

In [ ]:
# Write your code here for Option D (if selected)
# Test T = 10, 20, 30, 50 and record MAE/RMSE for each

In [ ]:
# --- Save your results ---
MAE_optionD_T10, RMSE_optionD_T10 = evaluate(test_matrix, YOUR_PREDICTED_MATRIX)
MAE_optionD_T20, RMSE_optionD_T20 = evaluate(test_matrix, YOUR_PREDICTED_MATRIX)
MAE_optionD_T30, RMSE_optionD_T30 = evaluate(test_matrix, YOUR_PREDICTED_MATRIX)
MAE_optionD_T50, RMSE_optionD_T50 = evaluate(test_matrix, YOUR_PREDICTED_MATRIX)

In [ ]:
# Please don't change this cell

print("=" * 70)
print("Option D: Significance Weighting")
print("=" * 70)
print(f"T = 10:  MAE = {MAE_optionD_T10},  RMSE = {RMSE_optionD_T10}")
print(f"T = 20:  MAE = {MAE_optionD_T20},  RMSE = {RMSE_optionD_T20}")
print(f"T = 30:  MAE = {MAE_optionD_T30},  RMSE = {RMSE_optionD_T30}")
print(f"T = 50:  MAE = {MAE_optionD_T50},  RMSE = {RMSE_optionD_T50}")
print(f"Baseline (no weighting):  MAE = {MAE_task1},  RMSE = {RMSE_task1}")

---
# Summary of All Results

In [ ]:
# Please don't change this cell
print("=" * 70)
print("SUMMARY")
print("=" * 70)
print()
print(f"Task 1 Baseline:  MAE = {MAE_task1},  RMSE = {RMSE_task1}")
print()
print(f"Selected options: {OPTION_1}, {OPTION_2}")
print()

if OPTION_1 == 'A' or OPTION_2 == 'A':
    print("Option A — Similarity Measure Comparison:")
    print(f"  Cosine:           MAE = {MAE_optionA_cosine},  RMSE = {RMSE_optionA_cosine}")
    print(f"  Pearson:          MAE = {MAE_optionA_pearson},  RMSE = {RMSE_optionA_pearson}")
    print()

if OPTION_1 == 'B' or OPTION_2 == 'B':
    print("Option B — Prediction Method Comparison:")
    print(f"  Weighted sum:     MAE = {MAE_optionB},  RMSE = {RMSE_optionB}")
    print()
    
if OPTION_1 == 'C' or OPTION_2 == 'C':
    print("Option C — Neighbourhood Size Tuning:")
    print(f"  k =  5:  MAE = {MAE_optionC_k5},  RMSE = {RMSE_optionC_k5}")
    print(f"  k = 10:  MAE = {MAE_optionC_k10},  RMSE = {RMSE_optionC_k10}")
    print(f"  k = 20:  MAE = {MAE_optionC_k20},  RMSE = {RMSE_optionC_k20}")
    print(f"  k = 30:  MAE = {MAE_optionC_k30},  RMSE = {RMSE_optionC_k30}")
    print(f"  k = 50:  MAE = {MAE_optionC_k50},  RMSE = {RMSE_optionC_k50}")
    print()
    
if OPTION_1 == 'D' or OPTION_2 == 'D':
    print("Option D — Significance Weighting:")
    print(f"  T = 10:  MAE = {MAE_optionD_T10},  RMSE = {RMSE_optionD_T10}")
    print(f"  T = 20:  MAE = {MAE_optionD_T20},  RMSE = {RMSE_optionD_T20}")
    print(f"  T = 30:  MAE = {MAE_optionD_T30},  RMSE = {RMSE_optionD_T30}")
    print(f"  T = 50:  MAE = {MAE_optionD_T50},  RMSE = {RMSE_optionD_T50}")
    print()



The baseline item-based collaborative filtering model using adjusted cosine similarity and KNN regression (k = 20) achieved an MAE of 0.7501 and an RMSE of 0.9611. 

In option A, adjusted cosine similarity out performed both cosine similarity  and pearson correlation, indicating that accountibg for user rating bias improves recommendation accuracy. Pearson correlation  produced the highest  error values and was therefore the least effective similarity measure for this dataset.

In option C, different neighbourhood sizes were evaluated. The lowest MAE was achieved by the baseline model with k= 20, while the lowest RMSE was achieved at k=30.

However, the differences between k=20, k=30 and k=50 were very small, indicating that the recommender system is relatively stable across larger neighbourhood sizes. 

Overall, adjusted cosine similarity combined with KNN regression provided the best balance between accuracy and consistency and remains the preferred configuration for this dataset.
